# Method 2 completion 5: stress
Attach the strict completion bundle. See docs/method2_completion.md for the required previous outputs. GPU T4, fresh session. Test results must not select hyperparameters.


In [1]:
from pathlib import Path
import os, shutil, subprocess, sys, json
from pathlib import Path
import hashlib, json

def file_digest(path: Path) -> str:
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

def choose_bundle(input_root: Path) -> Path:
    candidates = []
    for manifest in sorted(input_root.rglob("completion_manifest.json")):
        data = json.loads(manifest.read_text(encoding="utf-8"))
        if data.get("files") and all((manifest.parent / name).is_file() for name in data["files"]):
            candidates.append(manifest)
    if not candidates or len({file_digest(p) for p in candidates}) != 1:
        raise RuntimeError(f"Attach one complete bundle identity. Complete candidates: {[str(p) for p in candidates]}")
    print("Bundle selected:", candidates[0].parent)
    return candidates[0].parent

def choose_stage(input_root: Path, stage: str, bundle_digest: str) -> Path:
    candidates = []
    identities = set()
    for marker in sorted(input_root.rglob(f"{stage}_completed.json")):
        if marker.parts[-4:-1] != ("results", "method2", "completion"):
            continue
        root = marker.parents[3]
        data = json.loads(marker.read_text(encoding="utf-8"))
        hashes = data.get("checkpoint_hashes", {})
        if data.get("stage") != stage or data.get("completed") is not True or not hashes:
            continue
        if data.get("bundle_manifest_sha256") != bundle_digest:
            print("Skipping marker from another bundle:", marker)
            continue
        invalid = [name for name, digest in hashes.items()
                   if not (root / name).is_file() or file_digest(root / name) != digest]
        if invalid:
            print("Skipping incomplete or mismatched stage output:", marker, invalid[:3])
            continue
        candidates.append(marker)
        identities.add(json.dumps(hashes, sort_keys=True))
    if not candidates or len(identities) != 1:
        raise RuntimeError(f"Attach one complete {stage} checkpoint identity. Verified candidates: {[str(p) for p in candidates]}")
    print(f"{stage} input selected:", candidates[0])
    return candidates[0]


os.environ["CUDA_VISIBLE_DEVICES"] = "0"
bundle = choose_bundle(Path("/kaggle/input"))
work = Path("/kaggle/working")
for name in ["src", "scripts", "configs", "data", "docs", "notebooks"]:
    shutil.copytree(bundle / name, work / name, dirs_exist_ok=True)
shutil.copy2(bundle / "same_domain_feasibility.json", work / "same_domain_feasibility.json")
shutil.copy2(bundle / "completion_manifest.json", work / "completion_manifest.json")
os.chdir(work)
caches = list(Path("/kaggle/input").rglob("models--BAAI--bge-m3"))
if caches:
    cache_hub = caches[0].parent
    os.environ["HF_HUB_CACHE"] = str(cache_hub)
    os.environ["HF_HOME"] = str(cache_hub.parent)
pins = json.loads(Path("configs/method2/pinned_versions.json").read_text())["pinned"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *[f"{k}=={v}" for k,v in pins.items()], "jsonschema", "pyyaml", "matplotlib", "rank_bm25", "datasets", "accelerate"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
lora_smoke = """import torch
from transformers import XLMRobertaConfig, XLMRobertaModel
from peft import LoraConfig
config = XLMRobertaConfig(vocab_size=32, hidden_size=8, num_hidden_layers=1, num_attention_heads=2, intermediate_size=16, max_position_embeddings=32)
model = XLMRobertaModel(config)
model.add_adapter(LoraConfig(r=2, lora_alpha=4, target_modules=["query", "value"]))
model(torch.tensor([[0, 5, 2]])).last_hidden_state.square().mean().backward()
assert any(p.grad is not None for n, p in model.named_parameters() if "lora_" in n)
print("LoRA environment smoke passed")
"""
subprocess.run([sys.executable, "-c", lora_smoke], check=True)


Bundle selected: /kaggle/input/datasets/dathq12/output-method2-completion-04-evaluation
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.6/739.6 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 32.1 MB/s eta 0:00:00
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
LoRA environment smoke passed


CompletedProcess(args=['/usr/bin/python3', '-c', 'import torch\nfrom transformers import XLMRobertaConfig, XLMRobertaModel\nfrom peft import LoraConfig\nconfig = XLMRobertaConfig(vocab_size=32, hidden_size=8, num_hidden_layers=1, num_attention_heads=2, intermediate_size=16, max_position_embeddings=32)\nmodel = XLMRobertaModel(config)\nmodel.add_adapter(LoraConfig(r=2, lora_alpha=4, target_modules=["query", "value"]))\nmodel(torch.tensor([[0, 5, 2]])).last_hidden_state.square().mean().backward()\nassert any(p.grad is not None for n, p in model.named_parameters() if "lora_" in n)\nprint("LoRA environment smoke passed")\n'], returncode=0)

In [2]:
prior_stage = 'evaluation'
if prior_stage:
    prior_marker = choose_stage(Path("/kaggle/input"), prior_stage, file_digest(work / "completion_manifest.json"))
    previous = prior_marker.parents[3]
    for name in ["artifacts/method2", "results/method2/completion", "data/method2/index"]:
        if (previous / name).exists():
            shutil.copytree(previous / name, work / name, dirs_exist_ok=True, ignore=shutil.ignore_patterns("checkpoint-*"))
if False:
    validation_marker = choose_stage(Path("/kaggle/input"), "validation", file_digest(work / "completion_manifest.json"))
    validation_root = validation_marker.parents[3]
    shutil.copytree(validation_root / "results/method2/completion/validation", work / "results/method2/completion/validation", dirs_exist_ok=True)
    shutil.copy2(validation_marker, work / "results/method2/completion/validation_completed.json")
    shutil.copytree(validation_root / "artifacts/method2/crossencoder", work / "artifacts/method2/crossencoder", dirs_exist_ok=True, ignore=shutil.ignore_patterns("checkpoint-*"))
ce = Path("artifacts/method2/crossencoder/run01/final")
if True and not (ce / "crossencoder_heads.pt").exists():
    heads = [p for p in Path("/kaggle/input").rglob("crossencoder_heads.pt") if p.parent.name == "final"]
    if len(heads) != 1:
        raise RuntimeError(f"Attach one CE final checkpoint (model + heads + tokenizer), found {len(heads)}")
    shutil.copytree(heads[0].parent, ce, dirs_exist_ok=True)


Skipping incomplete or mismatched stage output: /kaggle/input/datasets/dathq12/output-method2-completion-04-evaluation/method2_completion_evaluation_reports/results/method2/completion/evaluation_completed.json ['artifacts/method2/biencoder/strict_round1/final/tokenizer.json', 'artifacts/method2/biencoder/strict_round1/final/sentence_bert_config.json', 'artifacts/method2/biencoder/strict_round1/final/config_sentence_transformers.json']
evaluation input selected: /kaggle/input/datasets/dathq12/output-method2-completion-04-evaluation/results/method2/completion/evaluation_completed.json


In [3]:
subprocess.run([sys.executable, "scripts/method2/run_completion.py", "stress"], check=True)
print(Path("results/method2/completion/stress_completed.json").read_text(encoding="utf-8")[:2000])


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2338.75it/s]


[stress] → results/method2/completion/stress/stress_accuracy_random.png
[stress] → results/method2/completion/stress/stress_latency_random.png
{
  "stage": "stress",
  "completed": true,
  "bundle_manifest_sha256": "d562aa4b38bbba20eb28adfc1702df42c193add4795c7415528a62bde945c908",
  "device": "Tesla T4",
  "torch": "2.10.0+cu128",
  "warmup": "3 per evaluation mode; excluded from measurements",
  "checkpoint_hashes": {
    "artifacts/method2/biencoder/strict_round1/final/config_sentence_transformers.json": "9696cf0d420f420dd1da034bb7a6d750ccacb6a14a39ba11bebb5135fbbffdd1",
    "artifacts/method2/biencoder/strict_round1/final/tokenizer_config.json": "71ab1498315ee1915b6de5fff8eb889835df8e485ac40b20c1de29838301ed31",
    "artifacts/method2/biencoder/strict_round1/final/README.md": "8d54d696f1277e3cce3405d478f0bf361e1ec397c79650655da971331b48c6f2",
    "artifacts/method2/biencoder/strict_round1/final/sentence_bert_config.json": "3084164002c0bca01b0259c5327123803fce32e660a57feb93184ffead1

In [4]:
archive = work / "method2_completion_stress_reports.tar.gz"
items = [name for name in ["results/method2/completion", "data/method2/biencoder/train_mined.jsonl"] if Path(name).exists()]
subprocess.run(["tar", "czf", str(archive), *items], check=True)
print(archive, archive.stat().st_size)
print("Save the whole notebook output for the next stage; this small archive does not contain model weights or the index.")


/kaggle/working/method2_completion_stress_reports.tar.gz 11067445
Save the whole notebook output for the next stage; this small archive does not contain model weights or the index.
